Original Kookmin Competition Code! It will be used as reference for the Kookmin_2.py 

In [1]:
sch = 'linear'  
init_rate = 0.3 
init_method = 'snip_little_grad' 
lr = 0.001  
epoch = 5  
weight_decay = 5e-4

In [2]:
import os
import time
import subprocess

import pandas as pd
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision.models import resnet18
from torch.utils.data import DataLoader, Dataset

import numpy as np
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, Subset

from torch.nn import Conv2d
import torch.nn.utils.prune as prune

DEVICE = 'cuda'
if DEVICE != 'cuda':
    raise RuntimeError('Make sure you have added an accelerator to your notebook; the submission will fail otherwise!')

In [3]:
from torch.optim.lr_scheduler import _LRScheduler

class LinearAnnealingLR(_LRScheduler):
    def __init__(self, optimizer, num_annealing_steps, num_total_steps):
        self.num_annealing_steps = num_annealing_steps
        self.num_total_steps = num_total_steps

        super().__init__(optimizer)

    def get_lr(self):
        if self._step_count <= self.num_annealing_steps:
            return [base_lr * self._step_count / self.num_annealing_steps for base_lr in self.base_lrs]
        else:
            return [base_lr * (self.num_total_steps - self._step_count) / (self.num_total_steps - self.num_annealing_steps) for base_lr in self.base_lrs]
        
class IncreasingLR(_LRScheduler):
    def __init__(self, optimizer, num_total_steps):
        self.num_total_steps = num_total_steps

        super().__init__(optimizer)

    def get_lr(self):
        return [base_lr * self._step_count / self.num_total_steps for base_lr in self.base_lrs]


In [4]:
def mp_importance_score(model): #weight maginute
    score_dict = {}
    for m in model.modules():
        if isinstance(m, (Conv2d,)):
            score_dict[(m, "weight")] = m.weight.data.abs()
    return score_dict

def mp_prune(model, ratio):
    score_dict = mp_importance_score(model)

    prune.global_unstructured(
        parameters=score_dict.keys(),
        pruning_method=prune.L1Unstructured,
        amount=ratio,
        importance_scores=score_dict,
    )

def remove_prune(model):
    print("Remove hooks for multiplying masks (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            prune.remove(m, "weight")
            
def pruning_model_random(model, px):
    print("Apply Unstructured Random Pruning Globally (all conv layers)")
    parameters_to_prune = []
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            parameters_to_prune.append((m, "weight"))
    parameters_to_prune = tuple(parameters_to_prune)
    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.RandomUnstructured,
        amount=px,
    )

In [5]:
@torch.no_grad()
def re_init_model_random(model, px):
    print("Apply Unstructured random re init no grads Globally (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            prob = torch.rand_like(mask.float(), device=DEVICE)
            topk = torch.topk(prob.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            
            m.weight.data[mask] = nn.Conv2d(in_c, out_c, ke, device=DEVICE).weight[mask]
            
def re_init_model_random_diffrent_init(model, px):
    print("Apply Unstructured random re init no grads Globally (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            prob = torch.rand_like(mask.float(), device=DEVICE)
            topk = torch.topk(prob.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            
            temp_weight = torch.empty_like(mask, dtype=torch.float)
            nn.init.kaiming_normal_(temp_weight, mode="fan_out", nonlinearity="relu")
            m.weight.data[mask] = temp_weight[mask]

@torch.no_grad()
def re_init_model_mp(model, px): # re init smallest weight
    print("Apply Unstructured mp re init no grads Globally (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            value = -m.weight.abs()
            topk = torch.topk(value.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            
            m.weight.data[mask] = nn.Conv2d(in_c, out_c, ke, device=DEVICE).weight[mask]
            
@torch.no_grad()
def re_init_model_mp_ver2(model, px): # re init Biggest weight
    print("Apply Unstructured mp re init no grads Globally (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            value = m.weight.abs()
            topk = torch.topk(value.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            
            m.weight.data[mask] = nn.Conv2d(in_c, out_c, ke, device=DEVICE).weight[mask]
            
@torch.no_grad()
def re_init_model_mp(model, px): # re init smallest weight
    print("Apply Unstructured mp re init no grads Globally (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            value = -m.weight.abs()
            topk = torch.topk(value.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            
            m.weight.data[mask] = nn.Conv2d(in_c, out_c, ke, device=DEVICE).weight[mask]

@torch.no_grad()
def re_init_model_snip(model, px): # re init Biggest gradients
    print("Apply Unstructured mp re init no grads Globally (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            value = m.weight.grad.abs()
            topk = torch.topk(value.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            
            m.weight.data[mask] = nn.Conv2d(in_c, out_c, ke, device=DEVICE).weight[mask]

@torch.no_grad()
def re_init_model_snip_ver2(model, px): # re init smallest gradients
    print("Apply Unstructured snip ve2 re init no grads Globally (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            value = -m.weight.grad.abs()
            topk = torch.topk(value.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            
            m.weight.data[mask] = nn.Conv2d(in_c, out_c, ke, device=DEVICE).weight[mask]

In [6]:
def get_grads_for_snip(model, retain_loader, forget_loader):
    indices = torch.randperm(len(retain_loader.dataset), dtype=torch.int32, device='cpu')[:len(forget_loader.dataset)]
    retain_dataset = Subset(retain_loader.dataset, indices)
    retain_loader = DataLoader(retain_dataset, batch_size=64, shuffle=True)
    
    model.zero_grad()
    for sample in retain_loader:
        inputs = sample["image"]
        targets = sample["age_group"]
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        outputs = model(inputs)
        loss = F.cross_entropy(outputs, targets)
        loss.backward()
        
    for sample in forget_loader:
        inputs = sample["image"]
        targets = sample["age_group"]
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        outputs = model(inputs)
        loss = -F.cross_entropy(outputs, targets)
        loss.backward()
        
def get_all_grads_for_snip(model, retain_loader, forget_loader):
    retain_weight = len(forget_loader.dataset) / len(retain_loader.dataset)
    model.zero_grad()
    for sample in retain_loader:
        inputs = sample["image"]
        targets = sample["age_group"]
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        outputs = model(inputs)
        loss = retain_weight * F.cross_entropy(outputs, targets)
        loss.backward()
        
    for sample in forget_loader:
        inputs = sample["image"]
        targets = sample["age_group"]
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        outputs = model(inputs)
        loss = -F.cross_entropy(outputs, targets)
        loss.backward()

In [7]:
from functools import partial
from types import MethodType

def zero_grads(mask, g):
    return g*mask

def zero_grads_ver2(mask, g):
    out = g.clone()
    out[~mask] = 0
    return out

def zero_grads_ver3(self, g):
    out = g.clone()
    out[~self.mask] = 0
    return out

@torch.no_grad()
def re_init_model_random_zero_grad(model, px):
    print('re_init_model_random_zero_grad')
    print("Apply Unstructured random re init no grads Globally (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            prob = torch.rand_like(mask.float(), device=DEVICE)
            topk = torch.topk(prob.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            
            m.weight.data[mask] = nn.Conv2d(in_c, out_c, ke, device=DEVICE).weight[mask]
            m.weight.register_hook(partial(zero_grads, mask))

@torch.no_grad()
def re_init_model_random_zero_grad_ver2(model, px):
    print('re_init_model_random_zero_grad_ver2')
    print("Apply Unstructured random re init no grads Globally (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            prob = torch.rand_like(mask.float(), device=DEVICE)
            topk = torch.topk(prob.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            
            m.weight.data[mask] = nn.Conv2d(in_c, out_c, ke, device=DEVICE).weight[mask]
            m.weight.register_hook(partial(zero_grads_ver2, mask))

@torch.no_grad()
def re_init_model_random_zero_grad_ver3(model, px):
    print('re_init_model_random_zero_grad_ver3')
    print("Apply Unstructured random re init no grads Globally (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            prob = torch.rand_like(mask.float(), device=DEVICE)
            topk = torch.topk(prob.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            m.weight.data[mask] = nn.Conv2d(in_c, out_c, ke, device=DEVICE).weight[mask]

            m.mask = mask
            m.backward_fn = MethodType(zero_grads_ver3, m)
        
            m.weight.register_hook(m.backward_fn)

In [8]:
class Masker(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, mask):
        ctx.save_for_backward(mask)
        return x

    @staticmethod
    def backward(ctx, grad):
        mask, = ctx.saved_tensors
        return grad * mask, None
    
    
class MaskConv2d(nn.Conv2d):
    def __init__(self, mask, in_channels, out_channels, kernel_size, stride=1,
                 padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device='cpu'):
        super(MaskConv2d, self).__init__(in_channels, out_channels, kernel_size, stride,
                                     padding, dilation, groups, bias, padding_mode, device=device)
        self.mask = mask

    def forward(self, input):
        masked_weight = Masker.apply(self.weight, self.mask)
        return super(MaskConv2d, self)._conv_forward(input, masked_weight, self.bias)
    
@torch.no_grad()
def re_init_model_random_zero_grad_Maskconv(model, px):
    print("Apply Unstructured random re init no grads Globally (all conv layers)")
    for name, m in list(model.named_modules()):
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            prob = torch.rand_like(mask.float(), device=DEVICE)
            topk = torch.topk(prob.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True

            new_conv = MaskConv2d(mask, m.in_channels, m.out_channels, m.kernel_size, m.stride,
                 m.padding, m.dilation, m.groups, m.bias!=None, m.padding_mode, device=DEVICE)
            #nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")
            new_conv.weight.data[~mask] = m.weight[~mask]
            setattr(model, name, new_conv)
            
@torch.no_grad()
def re_init_model_random_little_grad_Maskconv_fanout(model, px):
    print("Apply Unstructured re_init_model_random_little_grad_Maskconv_fanout (all conv layers)")
    for name, m in list(model.named_modules()):
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            prob = torch.rand_like(mask.float(), device=DEVICE)
            topk = torch.topk(prob.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            grad_mask = mask.clone().float()
            grad_mask[grad_mask==0] += 0.1

            new_conv = MaskConv2d(grad_mask, m.in_channels, m.out_channels, m.kernel_size, m.stride,
                 m.padding, m.dilation, m.groups, m.bias!=None, m.padding_mode, device=DEVICE)
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")
            new_conv.weight.data[~mask] = m.weight[~mask]
            setattr(model, name, new_conv)

In [9]:
@torch.no_grad()
def re_init_model_snip_ver2_zero_grad(model, px): # re init smallest gradients
    print("Apply Unstructured re_init_model_snip_ver2_zero_grad Globally (all conv layers)")
    for name, m in list(model.named_modules()):
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            value = -m.weight.grad.abs()
            topk = torch.topk(value.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True

            new_conv = MaskConv2d(mask, m.in_channels, m.out_channels, m.kernel_size, m.stride,
                 m.padding, m.dilation, m.groups, m.bias!=None, m.padding_mode, device=DEVICE)
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")

            new_conv.weight.data[~mask] = m.weight[~mask]
            set_layer(model, name, new_conv)

@torch.no_grad()
def re_init_model_snip_ver2_little_grad(model, px): # re init smallest gradients
    print("Apply Unstructured re_init_model_snip_ver2_little_grad Globally (all conv layers)")
    for name, m in list(model.named_modules()):
        if isinstance(m, nn.Conv2d):
            mask = torch.zeros_like(m.weight, device=DEVICE).bool()
            nparams_toprune = round(px*mask.nelement())
        
            out_c, in_c, ke, _ = mask.shape
            value = -m.weight.grad.abs()
            topk = torch.topk(value.view(-1), k=nparams_toprune)
            mask.view(-1)[topk.indices] = True
            grad_mask = mask.clone().float()
            grad_mask[grad_mask==0] += 0.1

            new_conv = MaskConv2d(grad_mask, m.in_channels, m.out_channels, m.kernel_size, m.stride,
                 m.padding, m.dilation, m.groups, m.bias!=None, m.padding_mode, device=DEVICE)
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")

            new_conv.weight.data[~mask] = m.weight[~mask]

            set_layer(model, name, new_conv)

def set_layer(model, layer_name, layer):
    splited = layer_name.split('.')
    if len(splited) == 1:
        setattr(model, splited[0], layer)
    elif len(splited) == 3:
        setattr(getattr(model, splited[0])[int(splited[1])], splited[2], layer)
    elif len(splited) == 4:
        getattr(getattr(model, splited[0])[int(splited[1])], splited[2])[int(splited[3])] = layer

@torch.no_grad()
def replace_maskconv(model):
    print("Remove Maskconv")
    for name, m in list(model.named_modules()):
        if isinstance(m, MaskConv2d):
            conv = nn.Conv2d(m.in_channels, m.out_channels, m.kernel_size, m.stride,
                 m.padding, m.dilation, m.groups, m.bias!=None, m.padding_mode, device=DEVICE)
            conv.weight.data = m.weight
            conv.bias = m.bias
            set_layer(model, name, conv)

In [10]:
@torch.no_grad()
def reinit_topn(model, n):
    print(f"Reinit top {n} conv2d(only 5 layer available)")
    target_layer = ['layer4.0.conv1', 'layer4.0.conv2', 'layer4.0.downsample.0','layer4.1.conv1', 'layer4.1.conv2']
    target_layer = target_layer[-n:]
    for name, m in list(model.named_modules()):
        if name in target_layer:
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

In [11]:
@torch.no_grad()
def reinit_batchnorm(model):
    print("Reinit BatchNorm2d")
    for name, m in list(model.named_modules()):
        if isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)


In [12]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

def snip_importance_score(model):
    score_dict = {}
    for m in model.modules():
        if isinstance(m, (nn.Conv2d,)):
            score_dict[(m, "weight")] = -m.weight.grad.abs()
    return score_dict

@torch.no_grad()
def prune_model_snip_ver2(model, px): # re init smallest gradients
    print("Apply Unstructured snip ver2 prune grads Globally (all conv layers)")

    score_dict = snip_importance_score(model)
    prune.global_unstructured(
        parameters=score_dict.keys(),
        pruning_method=prune.L1Unstructured,
        amount=px,
        importance_scores=score_dict,
    )

def remove_prune(model):
    print("Remove hooks for multiplying masks (all conv layers)")
    for name, m in model.named_modules():
        if isinstance(m, nn.Conv2d):
            prune.remove(m, "weight")

In [13]:
def load_example(df_row):
    image = torchvision.io.read_image(df_row['image_path'])
    result = {
        'image': image,
        'image_id': df_row['image_id'],
        'age_group': df_row['age_group'],
        'age': df_row['age'],
        'person_id': df_row['person_id']
    }
    return result


class HiddenDataset(Dataset):
    '''The hidden dataset.'''
    def __init__(self, split='train'):
        super().__init__()
        self.examples = []

        df = pd.read_csv(f'/kaggle/input/neurips-2023-machine-unlearning/{split}.csv')
        df['image_path'] = df['image_id'].apply(
            lambda x: os.path.join('/kaggle/input/neurips-2023-machine-unlearning/', 'images', x.split('-')[0], x.split('-')[1] + '.png'))
        df = df.sort_values(by='image_path')
        df.apply(lambda row: self.examples.append(load_example(row)), axis=1)
        if len(self.examples) == 0:
            raise ValueError('No examples.')

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        example = self.examples[idx]
        image = example['image']
        image = image.to(torch.float32)
        example['image'] = image
        return example


def get_dataset(batch_size):
    '''Get the dataset.'''
    retain_ds = HiddenDataset(split='retain')
    forget_ds = HiddenDataset(split='forget')
    val_ds = HiddenDataset(split='validation')

    retain_loader = DataLoader(retain_ds, batch_size=batch_size, shuffle=True)#, num_workers=4)
    forget_loader = DataLoader(forget_ds, batch_size=batch_size, shuffle=True)#, num_workers=4)
    validation_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=True)#, num_workers=4)

    return retain_loader, forget_loader, validation_loader

In [14]:
def unlearning(
    net, 
    retain_loader, 
    forget_loader, 
    val_loader):
    if init_method=='random':
        re_init_model_random(net, init_rate)
    elif init_method=='random_init2':
        re_init_model_random_diffrent_init(net, init_rate)
    elif init_method=='mp':
        re_init_model_mp(net, init_rate)
    elif init_method=='reversed_mp':
        re_init_model_mp_ver2(net, init_rate)
    elif init_method=='snip':
        get_grads_for_snip(net, retain_loader, forget_loader)
        re_init_model_snip_ver2(net, init_rate)
    elif init_method=='random_zero_grad':
        re_init_model_random_zero_grad(net, init_rate)
    elif init_method=='random_zero_grad_ver2':
        re_init_model_random_zero_grad_ver2(net, init_rate)
    elif init_method=='random_zero_grad_ver3':
        re_init_model_random_zero_grad_ver3(net, init_rate)
    elif init_method=='maskconv':
        re_init_model_random_zero_grad_Maskconv(net, init_rate)
    elif init_method=='little_grad':
        re_init_model_random_little_grad_Maskconv_fanout(net, init_rate)
    elif init_method=='snip_zero_grad':
        replace_maskconv(net)
        get_grads_for_snip(net, retain_loader, forget_loader)
        re_init_model_snip_ver2_zero_grad(net, init_rate)
    elif init_method=='snip_little_grad':  
        replace_maskconv(net)
        get_grads_for_snip(net, retain_loader, forget_loader)
        re_init_model_snip_ver2_little_grad(net, init_rate)
    elif init_method=='snip_and_reinit_bn':  
        get_grads_for_snip(net, retain_loader, forget_loader)
        re_init_model_snip_ver2(net, init_rate)
        reinit_batchnorm(net)
    elif init_method=='reinit_bn':  
        reinit_batchnorm(net)
    elif init_method=='snip_and_topn_reinit':  
        get_grads_for_snip(net, retain_loader, forget_loader)
        re_init_model_snip_ver2(net, init_rate)
        reinit_topn(net, 1)
    elif init_method=='prune_snip':
        get_grads_for_snip(net, retain_loader, forget_loader)
        prune_model_snip_ver2(net, init_rate)
    elif init_method=='all_snip_reinit':
        get_all_grads_for_snip(net, retain_loader, forget_loader)
        re_init_model_snip_ver2(net, init_rate)
    else:
        raise "not implemented"
        
    """Simple unlearning by finetuning."""
    epochs = epoch
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr,
                      momentum=0.9, weight_decay=weight_decay)
    if sch=='cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif sch=='linear':
        scheduler = LinearAnnealingLR(optimizer, num_annealing_steps=(epochs+1)//2, num_total_steps=epochs+1)
    elif sch=='increase':
        scheduler = IncreasingLR(optimizer, num_total_steps=epochs)
    elif sch=='decrease':
        print('decrease')
        scheduler = LinearAnnealingLR(optimizer, num_annealing_steps=1, num_total_steps=epochs+1)
    
    net.train()

    for ep in range(epochs):
        for sample in retain_loader:
            inputs = sample["image"]
            targets = sample["age_group"]
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, targets)
            
            loss.backward()
            optimizer.step()
    
        scheduler.step()
    #remove_prune(net)  
    net.eval()

In [15]:
class DataSet(Dataset):
    def __init__(self, dataset, indices=None, is_train=True):
        if indices is not None:
            self.dataset = Subset(getattr(torchvision.datasets, 'CIFAR10')(train=is_train, root='./data', download=True), indices)
            self.targets = np.array(self.dataset.dataset.targets)[indices]
        else:
            self.dataset = getattr(torchvision.datasets,'CIFAR10')(train=is_train, root='./data', download=True)
            self.targets = np.array(self.dataset.targets)

        self.transfrom = self._get_cifar_transforms(is_train)

    def __len__(self):
        return len(self.targets)
    
    def __getitem__(self, index):
        data = self.dataset.__getitem__(index)
        return {'image' : self.transfrom(data[0]), 
                'age_group' : self.targets[index]}

    def _get_cifar_transforms(self, is_train=True):
        if is_train:
            transform_augment = transforms.Compose([
                transforms.Pad(padding=4, fill=(125,123,113)),
                transforms.RandomCrop(32, padding=0),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
            ])
            return transform_augment
        
        else:
            transform_test = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
            ])
            return transform_test

In [16]:
if os.path.exists('/kaggle/input/neurips-2023-machine-unlearning/empty.txt'):
    start = time.time()
    net = resnet18(weights=None, num_classes=10)
    net.to(DEVICE)
    from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
    train_dataset = DataSet(dataset='', is_train=True) # only using for StratifiedShuffleSplit
    cv = StratifiedShuffleSplit(n_splits=1, test_size=10000)
    train_index, valid_index = next(iter(cv.split(train_dataset, train_dataset.targets)))

    train_dataset = Subset(DataSet(dataset='', indices=train_index, is_train=True), range(1000))
    valid_dataset = DataSet(dataset='', indices=valid_index, is_train=True)

    train_loader = DataLoader(
            train_dataset, batch_size=64, shuffle=True, #num_workers=4, #pin_memory=True
        )
    os.makedirs('/kaggle/tmp', exist_ok=True)
    for i in range(2):
        unlearning(net, train_loader, train_loader, train_loader)
        state = net.state_dict()
        net2 = resnet18(weights=None, num_classes=10)
        print(net2.load_state_dict(state))
        print(time.time() - start)
        torch.save(state, f'/kaggle/tmp/unlearned_checkpoint_{i}.pth')
    #subprocess.run('touch submission.zip', shell=True)
    unlearned_ckpts = os.listdir('/kaggle/tmp')
    #subprocess.run('touch submission.zip', shell=True)
    subprocess.run('zip submission.zip /kaggle/tmp/*.pth', shell=True)
else:
    
    # Note: it's really important to create the unlearned checkpoints outside of the working directory 
    # as otherwise this notebook may fail due to running out of disk space.
    # The below code saves them in /kaggle/tmp to avoid that issue.
    
    os.makedirs('/kaggle/tmp', exist_ok=True)
    retain_loader, forget_loader, validation_loader = get_dataset(64)
    net = resnet18(weights=None, num_classes=10)
    net.to(DEVICE)
    for i in range(512):
        net.load_state_dict(torch.load('/kaggle/input/neurips-2023-machine-unlearning/original_model.pth'))
        unlearning(net, retain_loader, forget_loader, validation_loader)
        state = net.state_dict()
        torch.save(state, f'/kaggle/tmp/unlearned_checkpoint_{i}.pth')
        
    # Ensure that submission.zip will contain exactly 512 checkpoints 
    # (if this is not the case, an exception will be thrown).
    unlearned_ckpts = os.listdir('/kaggle/tmp')
    if len(unlearned_ckpts) != 512:
        raise RuntimeError('Expected exactly 512 checkpoints. The submission will throw an exception otherwise.')
        
    subprocess.run('zip submission.zip /kaggle/tmp/*.pth', shell=True)

OSError: [Errno 30] Read-only file system: '/kaggle'